In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam

# Set visualization styles for a professional look
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

In [ ]:
# Seed configuration for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Generating synthetic enterprise HR dataset...")
num_records = 2500

# Base feature generation
data = {
    'MonthlyIncome': np.random.randint(3500, 18000, num_records),
    'OvertimeHours': np.random.randint(0, 60, num_records),
    'Age': np.random.randint(21, 65, num_records),
    'YearsAtCompany': np.random.randint(1, 15, num_records),
    'SatisfactionScore': np.random.choice([1, 2, 3, 4, 5], size=num_records, p=[0.1, 0.2, 0.3, 0.3, 0.1])
}
df = pd.DataFrame(data)

# High-end Feature Engineering: Creating composite interaction metrics
# Burnout Risk Index (Interaction between low satisfaction, high overtime, and lower relative income)
df['BurnoutIndex'] = (df['OvertimeHours'] * (6 - df['SatisfactionScore'])) / (df['MonthlyIncome'] / 1000)

# Establish complex non-linear mathematical dependencies for Attrition probability
log_odds = (
    0.05 * df['OvertimeHours']
    - 0.0003 * df['MonthlyIncome']
    - 0.4 * df['SatisfactionScore']
    + 0.15 * df['BurnoutIndex']
    - 0.02 * (df['Age'] - 35)
)
prob = 1 / (1 + np.exp(-log_odds))

# Introduce probabilistic binary classification labels with realistic noise
df['Attrition'] = np.random.binomial(1, prob)

print(f"Dataset generated. Target class distribution:\n{df['Attrition'].value_counts(normalize=True)}")

In [ ]:
# Feature matrix containing engineered interaction traits
feature_cols = ['MonthlyIncome', 'OvertimeHours', 'Age', 'YearsAtCompany', 'SatisfactionScore', 'BurnoutIndex']
X = df[feature_cols]
y = df['Attrition']

# Stratified split ensures the target ratio is perfectly balanced across sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Fit production scaling pipeline
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training shapes: {X_train_scaled.shape}, Testing shapes: {X_test_scaled.shape}")

In [ ]:
# Instantiate clean layer architecture avoiding legacy input warnings
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(16, activation='relu'),

    Dense(1, activation='sigmoid')
])

# Professional learning rate schedules & optimization bounds
custom_optimizer = Adam(learning_rate=0.005)
model.compile(optimizer=custom_optimizer, loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Adaptive dynamic learning rate constraints
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=0.00001, verbose=1)
stop_early = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# Handle potential class imbalance mathematically
neg, pos = np.bincount(y_train)
class_weights = {0: 1.0, 1: float(neg / pos)}

print("Executing neural network training optimization loop...")
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=60,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[reduce_lr, stop_early],
    verbose=1
)

In [ ]:
# Generate structural inferences
y_pred_probs = model.predict(X_test_scaled)
y_pred = (y_pred_probs > 0.5).astype(int)

print("\n" + "="*50)
print("             ENTERPRISE EVALUATION METRICS")
print("="*50)
print(classification_report(y_test, y_pred))

# ---------------- PLOTTING VISUALIZATIONS ----------------
fig, axs = plt.subplots(2, 2, figsize=(16, 12))

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axs[0, 0], cbar=False)
axs[0, 0].set_title('Confusion Matrix Breakdown')
axs[0, 0].set_xlabel('Predicted Risk Class')
axs[0, 0].set_ylabel('True Risk Class')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_probs)
roc_auc = auc(fpr, tpr)
axs[0, 1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Area = {roc_auc:.2f}')
axs[0, 1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axs[0, 1].set_title('Receiver Operating Characteristic (ROC)')
axs[0, 1].set_xlabel('False Positive Rate')
axs[0, 1].set_ylabel('True Positive Rate')
axs[0, 1].legend(loc="lower right")

# 3. Optimization Training Loss Curve
axs[1, 0].plot(history.history['loss'], label='Training Loss')
axs[1, 0].plot(history.history['val_loss'], label='Validation Loss')
axs[1, 0].set_title('Model Loss Convergence History')
axs[1, 0].set_xlabel('Epochs')
axs[1, 0].set_ylabel('Loss Value')
axs[1, 0].legend()

# 4. AUC Performance Curve
axs[1, 1].plot(history.history['auc'], label='Training AUC')
axs[1, 1].plot(history.history['val_auc'], label='Validation AUC')
axs[1, 1].set_title('Model AUC Curve Progression')
axs[1, 1].set_xlabel('Epochs')
axs[1, 1].set_ylabel('AUC Score')
axs[1, 1].legend()

plt.tight_layout()
plt.show()